In [ ]:
from pathlib import Path
import os, sys
for project_root in (Path.cwd(), *Path.cwd().parents):
    if (project_root / 'regarch').is_dir():
        os.chdir(project_root)
        sys.path.insert(0, str(project_root))
        break
else:
    raise RuntimeError('Run this notebook from inside the RegArch project.')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from regarch import (
    FateMap, GRNInference, generate_newick, load_fate_map_topology,
    parse_edge_dict,
)
from regarch.simulation import (
    generate_atac_data, generate_descendant_grns, generate_expression_data,
    generate_root_grn, perturb_branch_lengths, prepare_expression_data,
)
from regarch.utils.data_io import (
    get_dynamic_networks, get_positive_interactions,
    newick_to_edge_length_dict,
)
from regarch.utils.metrics import (
    compute_auprc, compute_auroc, metric_perturbation_impact,
)

simulation_data = Path('examples/data/simulation')
(simulation_data / 'input').mkdir(parents=True, exist_ok=True)
(simulation_data / 'additional_input_data').mkdir(parents=True, exist_ok=True)
for directory in [
    'examples/results/simulation/additional_output/ground_truth_grn',
    'examples/results/simulation/additional_output/sergio_files',
    'examples/results/simulation/additional_output/expression_files',
    'examples/results/simulation/inferred_grns/interaction_format',
    'examples/results/simulation/evaluation_result',
]:
    Path(directory).mkdir(parents=True, exist_ok=True)
pd.DataFrame([
    ('grn0', 'grn1', 0.4), ('grn0', 'grn2', 0.1),
    ('grn1', 'grn3', 0.6), ('grn1', 'grn4', 0.6),
    ('grn2', 'grn5', 0.9), ('grn2', 'grn6', 0.9),
], columns=['Parent', 'Child', 'Length']).to_csv(
    simulation_data / 'additional_input_data/grn_dict.csv', index=False)
pd.DataFrame({'x': [f'target_{i}' for i in range(150)]}).to_csv(
    simulation_data / 'additional_input_data/target_genes_name.csv', index=False)
pd.DataFrame({'x': [f'regulator_{i}' for i in range(100)]}).to_csv(
    simulation_data / 'additional_input_data/regulatory_genes_name.csv', index=False)

# Set True to regenerate synthetic data and recompute every GRN.
RECOMPUTE_GRNS = False

In [ ]:
fate_map_path = 'examples/data/simulation/additional_input_data/grn_dict.csv'
edge_dict = load_fate_map_topology(fate_map_path)
edge = parse_edge_dict(edge_dict)
fate_map = FateMap(edge)
newick_format = generate_newick(fate_map)

In [ ]:
grns_path = 'examples/results/simulation/additional_output/ground_truth_grn'
if RECOMPUTE_GRNS: grn0 = generate_root_grn(150, 100, grns_path)
if RECOMPUTE_GRNS: descendant_grns = generate_descendant_grns(7, 150, 100, edge_dict, grns_path, grn0)

In [ ]:
atac_file_path = 'examples/data/simulation/input/atac_data.csv'
if RECOMPUTE_GRNS: atac_data = generate_atac_data(descendant_grns, 7, 150, 100, atac_file_path)

In [ ]:
sergio_files_path = 'examples/results/simulation/additional_output/sergio_files'
expr_matrix_path = 'examples/results/simulation/additional_output/expression_files'
if RECOMPUTE_GRNS: generate_expression_data(descendant_grns, 7, 150, 100, 300, sergio_files_path, expr_matrix_path, 1)

In [ ]:
expression_file_path = 'examples/data/simulation/input/expression_data.csv'

target_genes_name = pd.read_csv('examples/data/simulation/additional_input_data/target_genes_name.csv')['x'].tolist()
regulatory_genes_name = pd.read_csv('examples/data/simulation/additional_input_data/regulatory_genes_name.csv')['x'].tolist()
if RECOMPUTE_GRNS: prepare_expression_data(150, target_genes_name, regulatory_genes_name, expr_matrix_path, expression_file_path)

In [ ]:
saved_dir = 'examples/results/simulation/inferred_grns/inferred_grn_defult_params'

if RECOMPUTE_GRNS: grn_inference_result = GRNInference(atac_file_path, expression_file_path, fate_map, saved_dir)
if RECOMPUTE_GRNS: grn_inference_result.infer_grn(20)

dynamic_networks_dict = get_dynamic_networks(saved_dir, fate_map, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
path = fate_map.get_path('grn5')

# Extract networks for grn0, grn2, and grn5
network0 = dynamic_networks_dict['grn0']
network2 = dynamic_networks_dict['grn2']
network5 = dynamic_networks_dict['grn5']

# Assign gene names to columns and index
network0.columns = network2.columns = network5.columns = regulatory_genes_name
network0.index = network2.index = network5.index = target_genes_name

# Get gene interactions
gene0_interaction = get_positive_interactions(network0)
gene2_interaction = get_positive_interactions(network2)
gene5_interaction = get_positive_interactions(network5)

gene0_interaction.to_csv('examples/results/simulation/inferred_grns/interaction_format/gene0_interaction.csv', header=None)
gene2_interaction.to_csv('examples/results/simulation/inferred_grns/interaction_format/gene2_interaction.csv', header=None)
gene5_interaction.to_csv('examples/results/simulation/inferred_grns/interaction_format/gene5_interaction.csv', header=None)

In [ ]:
nodes = list(fate_map.nodes.keys())
auroc_list = []
auprc_list = []

for node_id in nodes:
    grn_infer = dynamic_networks_dict[node_id]
    grn_true = pd.read_csv('examples/results/simulation/additional_output/ground_truth_grn/'+node_id+'.csv')
    grn_true = grn_true.T
    auroc = compute_auroc(grn_infer, grn_true, 0.1)
    auprc = compute_auprc(grn_infer, grn_true, 0.1)
    print([node_id, auroc, auprc])
    auroc_list.append([node_id, auroc])
    auprc_list.append([node_id, auprc])
    
auroc_df = pd.DataFrame(auroc_list)
auroc_df.columns = ['node_id','value']
auprc_df = pd.DataFrame(auprc_list)
auprc_df.columns = ['node_id','value']
auprc_df.to_csv('examples/results/simulation/evaluation_result/auprc.csv', index=None)
auroc_df.to_csv('examples/results/simulation/evaluation_result/auroc.csv', index=None)

In [ ]:
saved_dir_pert005 = 'examples/results/simulation/inferred_grns/inferred_grn_pert005'
fate_map_pert005 = perturb_branch_lengths(newick_format, sigma=0.05, seed=1234)

if RECOMPUTE_GRNS: grn_inference_result_pert005 = GRNInference(atac_file_path, expression_file_path, fate_map_pert005, saved_dir_pert005)
if RECOMPUTE_GRNS: grn_inference_result_pert005.infer_grn(20)
grn_dict_pert005 = get_dynamic_networks(saved_dir_pert005, fate_map_pert005, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
saved_dir_pert01 = 'examples/results/simulation/inferred_grns/inferred_grn_pert01'
fate_map_pert01 = perturb_branch_lengths(newick_format, sigma=0.1, seed=1234)
if RECOMPUTE_GRNS: grn_inference_result_pert01 = GRNInference(atac_file_path, expression_file_path, fate_map_pert01, saved_dir_pert01)
if RECOMPUTE_GRNS: grn_inference_result_pert01.infer_grn(20)
grn_dict_pert01 = get_dynamic_networks(saved_dir_pert01, fate_map_pert01, 0.1, regulatory_genes_name, target_genes_name)

saved_dir_pert03 = 'examples/results/simulation/inferred_grns/inferred_grn_pert03'
fate_map_pert03 = perturb_branch_lengths(newick_format, sigma=0.3, seed=1234)
if RECOMPUTE_GRNS: grn_inference_result_pert03 = GRNInference(atac_file_path, expression_file_path, fate_map_pert03, saved_dir_pert03)
if RECOMPUTE_GRNS: grn_inference_result_pert03.infer_grn(20)
grn_dict_pert03 = get_dynamic_networks(saved_dir_pert03, fate_map_pert03, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
saved_dir_pert05 = 'examples/results/simulation/inferred_grns/inferred_grn_pert05'
fate_map_pert05 = perturb_branch_lengths(newick_format, sigma=0.5, seed=1234)
if RECOMPUTE_GRNS: grn_inference_result_pert05 = GRNInference(atac_file_path, expression_file_path, fate_map_pert05, saved_dir_pert05)
if RECOMPUTE_GRNS: grn_inference_result_pert05.infer_grn(20)
grn_dict_pert05 = get_dynamic_networks(saved_dir_pert05, fate_map_pert05, 0.1, regulatory_genes_name, target_genes_name)

saved_dir_pert07 = 'examples/results/simulation/inferred_grns/inferred_grn_pert07'
fate_map_pert07 = perturb_branch_lengths(newick_format, sigma=0.7, seed=1234)
if RECOMPUTE_GRNS: grn_inference_result_pert07 = GRNInference(atac_file_path, expression_file_path,fate_map_pert07, saved_dir_pert07)
if RECOMPUTE_GRNS: grn_inference_result_pert07.infer_grn(20)
grn_dict_pert07 = get_dynamic_networks(saved_dir_pert07, fate_map_pert07, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
newick_format_pert_topology = '(((grn3:0.6,grn4:0.6)grn1:0.4,grn5:0.9)grn2:0.1,grn6:1.0)grn0:0;'
saved_dir_pert_topology = 'examples/results/simulation/inferred_grns/inferred_grn_pert_topology'

edge_dict_pert_topology = newick_to_edge_length_dict(newick_format_pert_topology)
edge_pert_topology = parse_edge_dict(edge_dict_pert_topology)
fate_map_pert_topology = FateMap(edge_pert_topology)

if RECOMPUTE_GRNS: grn_inference_result_pert_str = GRNInference(atac_file_path, expression_file_path, fate_map_pert_topology, saved_dir_pert_topology)
if RECOMPUTE_GRNS: grn_inference_result_pert_str.infer_grn(20)
grn_dict_pert_topology = get_dynamic_networks(saved_dir_pert_topology, fate_map_pert_topology, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
jaccard_means = []
overlap_means = []
grn_pert = [grn_dict_pert005, grn_dict_pert01, grn_dict_pert03, grn_dict_pert05, grn_dict_pert07, grn_dict_pert_topology]
for id in range(len(grn_pert)):
    jaccard_means.append(metric_perturbation_impact(grn_pert[id], dynamic_networks_dict)[0])
    overlap_means.append(metric_perturbation_impact(grn_pert[id], dynamic_networks_dict)[1])

print(jaccard_means)
print(overlap_means)

In [ ]:
from matplotlib.font_manager import FontProperties
import matplotlib.patches as mpatches

perturbation_labels = ["0.05", "0.1", "0.3", "0.5", "0.7", "Topology"]
colors = ['#EFEFEF', '#DCDDDD', '#C9CACA', '#B5B5B6', '#9FA0A0', '#A3CA89']
metric_labels = ["Jaccard index", "Key regulator overlap"]

x = np.arange(len(metric_labels)) 
bar_width = 0.1
fig, ax = plt.subplots(figsize=(3.2, 1.6))

for i, (jac, over, color, label) in enumerate(zip(jaccard_means, overlap_means, colors, perturbation_labels)):
    ax.bar(x[0] + i * bar_width - 0.25, jac, width=bar_width, color=color)
    ax.bar(x[1] + i * bar_width - 0.25, over, width=bar_width, color=color)

ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=6, fontname='Arial')

ax.set_ylabel("GRN similarity", fontsize=6, fontname='Arial')
ax.set_ylim(0.5, 1.05)
ax.tick_params(axis='y', labelsize=5)
for label in ax.get_yticklabels():
    label.set_fontname('Arial')

legend_handles = [mpatches.Patch(color=col, label=lab) for col, lab in zip(colors, perturbation_labels)]
legend_font = FontProperties(family='Arial', size=5.5)
ax.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.0, 0.7),
    prop=legend_font,
    markerscale=0.8,
    handlelength=1.0,
    handleheight=0.8,
    labelspacing=0.3,
    borderpad=0.3,
    borderaxespad=0.4
)

ax.set_axisbelow(True)
ax.grid(True, axis='y', linestyle='--', alpha=0.7, linewidth=0.5)
for spine in ax.spines.values():
    spine.set_color('#666666')
ax.tick_params(axis='x', color='#666666')
ax.tick_params(axis='y', color='#666666')

output_path = 'examples/results/simulation/additional_output/figures'
filename = "pert_grouped.eps"
os.makedirs(output_path, exist_ok=True)
save_path = os.path.join(output_path, filename)
plt.tight_layout()
plt.savefig(save_path, format='eps', bbox_inches='tight')
plt.show()